In [ ]:
import pandas as pd
import numpy as np

# Path to your Excel file
file_path = "/Users/jaacabrera/Documents/Python Scripts/data_io/HR Main Data (2019-2025).xlsx"

# Load all sheets (2019–2025)
all_sheets = pd.read_excel(file_path, sheet_name=None)

# Normalize column names
for sheet_name, df in all_sheets.items():
    df.columns = df.columns.str.strip()

# Store resignation and join results
resignation_records = []
join_records = []

# Get sorted list of years (tabs)
years = sorted(all_sheets.keys())

# Compare each consecutive year
for i in range(len(years) - 1):
    year_current = years[i]
    year_next = years[i + 1]
    
    df_current = all_sheets[year_current]
    df_next = all_sheets[year_next]
    
    names_current = set(df_current["Full Name"].dropna().unique())
    names_next = set(df_next["Full Name"].dropna().unique())
    
    # Employees in current year but not in next → resigned
    resigned = names_current - names_next
    for emp in resigned:
        try:
            joined_year = df_current.loc[df_current["Full Name"] == emp, "Year Joined"].values[0]
        except Exception:
            joined_year = pd.NA
        
        try:
            resigned_date = pd.to_datetime(
                df_current.loc[df_current["Full Name"] == emp, "Year/Date resigned"].values[0],
                errors="coerce"
            )
        except Exception:
            resigned_date = pd.NaT
        
        if pd.notna(resigned_date):
            resigned_year = resigned_date.year
            resigned_month = resigned_date.month
            resigned_quarter = f"Q{((resigned_date.month - 1) // 3 + 1)}"
        else:
            resigned_year = int(year_next)
            resigned_month = np.random.randint(1, 13)  # random month 1–12
            resigned_quarter = f"Q{((resigned_month - 1) // 3 + 1)}"
        
        resignation_records.append({
            "Employee": emp,
            "Year Joined": joined_year,
            "Resigned Between": f"{year_current} → {year_next}",
            "Resigned Year": resigned_year,
            "Resigned Month": resigned_month,
            "Resigned Quarter": resigned_quarter
        })
    
    # Employees in next year but not in current → joined
    joined = names_next - names_current
    for emp in joined:
        try:
            joined_year = df_next.loc[df_next["Full Name"] == emp, "Year Joined"].values[0]
        except Exception:
            joined_year = year_next
        
        try:
            joined_date = pd.to_datetime(
                df_next.loc[df_next["Full Name"] == emp, "Date Joined"].values[0],
                errors="coerce"
            )
        except Exception:
            joined_date = pd.NaT
        
        if pd.notna(joined_date):
            joined_month = joined_date.month
            joined_quarter = f"Q{((joined_date.month - 1) // 3 + 1)}"
        else:
            joined_month = np.random.randint(1, 13)
            joined_quarter = f"Q{((joined_month - 1) // 3 + 1)}"
        
        join_records.append({
            "Employee": emp,
            "Joined Year": joined_year,
            "Joined Month": joined_month,
            "Joined Quarter": joined_quarter
        })

# Convert to DataFrames
resigned_df = pd.DataFrame(resignation_records)
joined_df = pd.DataFrame(join_records)

# --- Add readable month names and full dates ---
resigned_df["Resigned Month Name"] = resigned_df["Resigned Month"].apply(
    lambda x: pd.to_datetime(str(x), format="%m").strftime("%B")
)

# Create a full resignation date (using the first day of the month)
resigned_df["Resigned Date"] = pd.to_datetime(
    resigned_df["Resigned Year"].astype(str) + "-" + 
    resigned_df["Resigned Month"].astype(str).str.zfill(2) + "-01"
)
joined_df["Joined Month Name"] = joined_df["Joined Month"].apply(
    lambda x: pd.to_datetime(str(x), format="%m").strftime("%B")
)

# --- Summary counts (Year + Month) ---
summary_resigned = (
    resigned_df.groupby(["Resigned Year", "Resigned Month Name"])
    .size()
    .reset_index(name="Number of Resignations")
)

summary_joined = (
    joined_df.groupby(["Joined Year", "Joined Month Name"])
    .size()
    .reset_index(name="Number of Joins")
)

# --- Starting headcount for 2019 ---
df_2019 = all_sheets["2019"]
starting_headcount_2019 = df_2019["Full Name"].dropna().nunique()

# --- Resource Growth Tracker ---
growth_records = []
current_headcount = starting_headcount_2019

# Add baseline for 2019
growth_records.append({
    "Year": 2019,
    "Starting Headcount": starting_headcount_2019,
    "Joins": 0,
    "Resignations": 0,
    "Net Change": 0,
    "Ending Headcount": starting_headcount_2019,
    "Growth Rate (%)": 0.0
})

for i in range(1, len(years)):
    year = int(years[i])
    
    resignations = summary_resigned.loc[summary_resigned["Resigned Year"] == year, "Number of Resignations"]
    resignations = int(resignations.sum()) if not resignations.empty else 0
    
    joins = summary_joined.loc[summary_joined["Joined Year"] == year, "Number of Joins"]
    joins = int(joins.sum()) if not joins.empty else 0
    
    net_change = joins - resignations
    ending_headcount = current_headcount + net_change
    growth_rate = (net_change / current_headcount) * 100 if current_headcount > 0 else 0
    
    growth_records.append({
        "Year": year,
        "Starting Headcount": current_headcount,
        "Joins": joins,
        "Resignations": resignations,
        "Net Change": net_change,
        "Ending Headcount": ending_headcount,
        "Growth Rate (%)": round(growth_rate, 2)
    })
    
    current_headcount = ending_headcount

growth_df = pd.DataFrame(growth_records)

# --- Save to Excel (overwrite cleanly each run) ---
with pd.ExcelWriter("/Users/jaacabrera/Documents/Python Scripts/data_io/Resigned_Tracker_2019_2025.xlsx", mode="w") as writer:
    resigned_df.to_excel(writer, sheet_name="Resigned Employees", index=False)
    joined_df.to_excel(writer, sheet_name="Joined Employees", index=False)
    summary_resigned.to_excel(writer, sheet_name="Resignation Summary", index=False)
    summary_joined.to_excel(writer, sheet_name="Join Summary", index=False)
    growth_df.to_excel(writer, sheet_name="Resource Growth Tracker", index=False)

print("Resignation, join, and resource growth tracker created successfully!")


Resignation, join, and resource growth tracker created successfully!
